# Experimental Validation of Condensate-Mediated Enhancer-Promoter Contacts

This notebook tests predictions from the Goh et al. (2025) model against experimental Hi-C/Micro-C and RNA-seq data from mouse embryonic stem cells (mESCs).

## Scientific Question

The Goh et al. model predicts that **RNA gradients guide transcriptional condensates toward gene promoters**, which should increase enhancer-promoter contact frequencies. Specifically:

1. **Higher transcription -> steeper RNA gradient -> stronger condensate attraction -> more E-P contacts**
2. **This effect should be distance-dependent**: strongest at intermediate genomic distances (50-200 kb)
3. **Super-enhancers** (where condensates form) should show stronger effects than typical enhancers

## Data Sources

We use publicly available data from Hsieh et al. (2022) Nature Genetics:
- **Micro-C**: High-resolution chromatin contact maps (GEO: GSE130275, GSE186454)
- **RNA-seq**: Gene expression data from the same mESC line
- **ChIP-seq**: H3K27ac, MED1, BRD4 for identifying enhancers and super-enhancers

## Outline

1. Download and preprocess data
2. Build enhancer-promoter pair catalog
3. Extract contact frequencies and expression values
4. Test prediction: correlation between expression and contacts
5. Test prediction: distance-dependence of the effect
6. Compare super-enhancers vs typical enhancers

In [ ]:
# Standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Plotting settings
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 14
sns.set_style('whitegrid')

# Data directories
DATA_DIR = Path('../data/experimental')
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Imports successful!")
print(f"Data directory: {DATA_DIR.resolve()}")

---
## Part 1: Data Download and Setup

We need three types of data:
1. **Micro-C contact matrix** (`.mcool` or `.hic` format)
2. **Gene expression** (RNA-seq, processed TPM/FPKM values)
3. **Enhancer annotations** (from H3K27ac ChIP-seq peaks)

In [ ]:
# Data file paths
MICROC_FILE = DATA_DIR / 'mESC_MicroC.mcool'
RNASEQ_FILE  = DATA_DIR / 'mESC_expression.tsv'
ENHANCER_FILE = DATA_DIR / 'mESC_H3K27ac_peaks.bed'
GENE_ANNOTATION = DATA_DIR / 'mm39_genes_tss.bed'
SUPER_ENHANCER_FILE = DATA_DIR / 'mESC_super_enhancers.bed'

# Full GENCODE GTF — try project root first (1.1 GB, vM38, 21,760 protein-coding genes)
# then fall back to the smaller stub in data/experimental/
GTF_FILE = Path('../gencode.vM38.annotation.gtf')
if not GTF_FILE.exists():
    GTF_FILE = DATA_DIR / 'gencode.vM38.basic.annotation.gtf'
    print('WARNING: Full GTF not found at project root — using truncated stub (chr1 only).')
    print('         Results will be genome-wide only when the full GTF is present.')
else:
    print(f'Full GTF found: {GTF_FILE.resolve()}')

# Check which data files exist
print('\nData file status:')
for name, path in [('Micro-C', MICROC_FILE), ('RNA-seq', RNASEQ_FILE),
                    ('Enhancers', ENHANCER_FILE), ('Genes (TSS bed)', GENE_ANNOTATION),
                    ('Super-enhancers', SUPER_ENHANCER_FILE)]:
    status = 'Found' if path.exists() else 'Missing'
    print(f'  {name}: {status} ({path.name})')

if not MICROC_FILE.exists():
    print('\nWARNING: mESC_MicroC.mcool not found.')
    print('  Download from GEO GSE130275:')
    print('  https://ftp.ncbi.nlm.nih.gov/geo/series/GSE130nnn/GSE130275/suppl/GSE130275_mESC_WT_combined_1.3B.mcool')
    print('  Save as: data/experimental/mESC_MicroC.mcool')


In [ ]:
# Instructions for downloading data
download_instructions = """
=== DATA DOWNLOAD INSTRUCTIONS ===

1. MICRO-C DATA (from 4DN or GEO):
   - 4DN Portal: https://data.4dnucleome.org/
   - Search for "mESC Micro-C"
   - Download .mcool file
   - Save as: data/experimental/mESC_MicroC.mcool

2. RNA-SEQ DATA:
   - ENCODE: https://www.encodeproject.org/
   - Search: "RNA-seq mouse embryonic stem cell"
   - Download gene quantifications (TPM or FPKM)
   - Save as: data/experimental/mESC_expression.tsv

3. ENHANCER PEAKS (H3K27ac ChIP-seq):
   - ENCODE: Search "H3K27ac ChIP-seq mESC"
   - Download narrowPeak or bed file
   - Save as: data/experimental/mESC_H3K27ac_peaks.bed

4. GENE ANNOTATIONS:
   - GENCODE: https://www.gencodegenes.org/mouse/
   - Download GTF, extract TSS positions
   - Save as: data/experimental/mm10_genes.bed

5. SUPER-ENHANCER CALLS (optional):
   - dbSUPER: http://asntech.org/dbsuper/
   - Save as: data/experimental/mESC_super_enhancers.bed
"""
print(download_instructions)

---
## Part 2: Load and Process Data

In [ ]:
def load_gene_expression(filepath):
    """
    Load gene expression data.
    Handles ENSMUSG IDs (preferred) or numeric/Entrez IDs (fallback).
    Prefers pme_TPM (Bayesian posterior mean) over raw TPM when available.
    """
    df = pd.read_csv(filepath, sep='\t')
    df.columns = df.columns.str.lower().str.replace(' ', '_').str.replace('(s)', '', regex=False)

    # Choose best TPM column: pme_tpm > tpm > fpkm
    if 'pme_tpm' in df.columns:
        df['tpm'] = df['pme_tpm']
        tpm_source = 'pme_TPM (posterior mean)'
    elif 'tpm' in df.columns:
        tpm_source = 'TPM'
    elif 'fpkm' in df.columns:
        df['tpm'] = df['fpkm']
        tpm_source = 'FPKM'
    else:
        raise ValueError('No TPM/FPKM column found in expression file')

    df['log2_tpm'] = np.log2(df['tpm'] + 1)

    # Try ENSMUSG IDs first (standard GENCODE/ENCODE quantification)
    ensmusg = df[df['gene_id'].astype(str).str.startswith('ENSMUSG', na=False)].copy()
    if len(ensmusg) > 0:
        ensmusg['ensembl_id'] = ensmusg['gene_id'].astype(str).str.split('.').str[0]
        print(f'Loaded expression for {len(ensmusg)} ENSMUSG genes '
              f'({(ensmusg["tpm"] > 0).sum()} expressed) using {tpm_source}')
        return ensmusg

    # Fallback: numeric/Entrez IDs — keep all rows, flag for user
    print(f'WARNING: No ENSMUSG IDs found in expression file.')
    print(f'  gene_id values look like: {df["gene_id"].iloc[:3].tolist()}')
    print(f'  These appear to be Entrez/numeric IDs. A direct ENSMUSG match is not possible.')
    print(f'  Using gene_id as-is. Gene-expression merge will likely produce 0 matches.')
    print(f'  To fix: obtain the ENSMUSG-quantified expression file (see README).')
    df['ensembl_id'] = df['gene_id'].astype(str)
    print(f'Loaded {len(df)} genes ({(df["tpm"] > 0).sum()} expressed) using {tpm_source}')
    return df


def load_genes_from_gtf(gtf_path):
    """Load protein-coding gene TSS coordinates from a GENCODE GTF file."""
    print(f'Parsing GTF: {gtf_path} ...')
    genes = []
    with open(gtf_path, encoding='utf-8') as f:
        for line in f:
            if line.startswith('#'): continue
            fields = line.strip().split('\t')
            if len(fields) < 9: continue
            if fields[2] != 'gene': continue
            attrs = fields[8]
            if 'protein_coding' not in attrs: continue
            try:
                gene_id   = attrs.split('gene_id "')[1].split('"')[0].split('.')[0]
                gene_name = attrs.split('gene_name "')[1].split('"')[0]
            except IndexError:
                continue
            chrom  = fields[0]
            strand = fields[6]
            tss = int(fields[3]) if strand == '+' else int(fields[4])
            genes.append({
                'ensembl_id': gene_id, 'gene_name': gene_name,
                'chrom': chrom, 'tss': tss, 'strand': strand
            })
    df = pd.DataFrame(genes)
    print(f'Loaded {len(df):,} protein-coding genes from GTF ({df["chrom"].nunique()} chromosomes)')
    return df


def load_bed_file(filepath, names=None):
    """Load a BED file into a DataFrame."""
    if names is None:
        names = ['chrom', 'start', 'end', 'name', 'score', 'strand',
                 'signalValue', 'pValue', 'qValue', 'peak']
    with open(filepath) as f:
        n_cols = len(f.readline().strip().split('\t'))
    names = names[:n_cols]
    df = pd.read_csv(filepath, sep='\t', names=names, comment='#')
    print(f'Loaded {len(df):,} regions from {filepath.name}')
    return df


def load_microc_matrix(filepath, resolution=6400):
    """
    Load Micro-C contact matrix from an .mcool file.
    Handles both the modern 'resolutions/' format and the older zoom-level format
    (cooler <= 0.7.x, where zoom levels are stored as integer keys '0'..'N').
    Selects the resolution closest to the requested value.
    """
    try:
        import cooler, h5py
    except ImportError:
        print('Please install required packages: pip install cooler h5py')
        return None

    filepath = str(filepath)

    with h5py.File(filepath, 'r') as fh:
        top_keys = list(fh.keys())
        has_resolutions = 'resolutions' in top_keys

        if has_resolutions:
            # Modern multi-resolution format
            available = sorted([int(r) for r in fh['resolutions'].keys()])
            best = min(available, key=lambda r: abs(r - resolution))
            uri = f'{filepath}::resolutions/{best}'
            fmt = 'resolutions'
        else:
            # Old zoom-level format: file attrs map zoom key -> bin size in bp
            zoom_map = {k: int(v) for k, v in fh.attrs.items() if str(k).isdigit()}
            if not zoom_map:
                # Single-resolution cooler
                uri = filepath
                best = cooler.Cooler(filepath).binsize
                fmt = 'single'
            else:
                best_key = min(zoom_map.keys(), key=lambda k: abs(zoom_map[k] - resolution))
                best = zoom_map[best_key]
                uri = f'{filepath}::{best_key}'
                fmt = f'zoom (key={best_key})'

    clr = cooler.Cooler(uri)
    actual_res = clr.binsize
    print(f'Loaded Micro-C [{fmt}] at {actual_res:,} bp ({actual_res/1000:.1f} kb) resolution')
    if abs(actual_res - resolution) > resolution * 0.5:
        print(f'  NOTE: requested {resolution:,} bp; closest available is {actual_res:,} bp')
    print(f'  Chromosomes: {len(clr.chromnames)}, total bins: {clr.info["nbins"]:,}')
    return clr


In [ ]:
# Load all data
expression_df = load_gene_expression(RNASEQ_FILE)
enhancers_df  = load_bed_file(ENHANCER_FILE)

# Load gene annotations from full GTF (falls back to stub if full not found)
genes_gtf = load_genes_from_gtf(GTF_FILE)

# Auto-generate mm39_genes_tss.bed if it doesn't exist
if not GENE_ANNOTATION.exists():
    print(f'\nGenerating {GENE_ANNOTATION.name} from GTF ...')
    tss_out = genes_gtf[['chrom', 'tss', 'gene_name', 'ensembl_id', 'strand']].copy()
    tss_out.insert(2, 'tss_end', tss_out['tss'] + 1)
    tss_out.to_csv(GENE_ANNOTATION, sep='\t', header=False, index=False)
    print(f'  Saved {len(tss_out):,} TSS entries to {GENE_ANNOTATION}')

# Load Micro-C (6.4 kb resolution — closest available in this file)
if MICROC_FILE.exists():
    clr = load_microc_matrix(MICROC_FILE, resolution=6400)
else:
    clr = None
    print('Micro-C file not found — contact analysis will be skipped.')

# Merge gene coordinates with expression via ENSMUSG IDs
genes_df = genes_gtf.merge(
    expression_df[['ensembl_id', 'tpm', 'log2_tpm']].drop_duplicates('ensembl_id'),
    on='ensembl_id', how='inner'
)
expressed_genes = genes_df[genes_df['tpm'] > 0]
print(f'\nGenes with coordinates + expression: {len(genes_df):,} ({len(expressed_genes):,} expressed)')
print(f'Chromosomes covered: {genes_df["chrom"].nunique()}')

if len(genes_df) == 0:
    print('\nWARNING: 0 genes matched between GTF and expression file.')
    print('  This usually means the expression file uses numeric/Entrez IDs instead of ENSMUSG.')
    print('  Proceeding with all GTF genes and TPM=1 placeholder for E-P pair building.')
    genes_df = genes_gtf.copy()
    genes_df['tpm'] = 1.0
    genes_df['log2_tpm'] = 0.0

# Define super-enhancers from top 5% H3K27ac signal
signal_threshold = enhancers_df['signalValue'].quantile(0.95)
enhancers_df['is_super_enhancer'] = enhancers_df['signalValue'] >= signal_threshold
print(f'\nSuper-enhancer threshold (95th pctl): {signal_threshold:.2f}')
print(f'Super-enhancer peaks: {enhancers_df["is_super_enhancer"].sum():,} / {len(enhancers_df):,}')

# Cluster nearby SE peaks into regions (merge peaks within 12.5 kb)
se_regions_all = []
for chrom in enhancers_df[enhancers_df['is_super_enhancer']]['chrom'].unique():
    se_chrom = enhancers_df[(enhancers_df['chrom']==chrom) &
                             (enhancers_df['is_super_enhancer'])].sort_values('start')
    if len(se_chrom) == 0: continue
    current = {'chrom': chrom, 'start': se_chrom.iloc[0]['start'],
                'end': se_chrom.iloc[0]['end']}
    for _, pk in se_chrom.iloc[1:].iterrows():
        if pk['start'] - current['end'] < 12500:
            current['end'] = max(current['end'], pk['end'])
        else:
            se_regions_all.append(current.copy())
            current = {'chrom': chrom, 'start': pk['start'], 'end': pk['end']}
    se_regions_all.append(current)
se_regions_df = pd.DataFrame(se_regions_all)
print(f'Clustered into {len(se_regions_df):,} super-enhancer regions')


---
## Part 3: Build Enhancer-Promoter Pair Catalog

In [ ]:
def build_ep_pairs(genes_df, enhancers_df, se_regions_df,
                   min_distance=5000, max_distance=500000):
    """Build catalog of enhancer-promoter pairs using ENSMUSG-matched data."""
    ep_pairs = []
    
    for _, gene in genes_df.iterrows():
        if gene['tpm'] <= 0:
            continue
        
        tss = gene['tss']
        chrom = gene['chrom']
        chrom_enhancers = enhancers_df[enhancers_df['chrom'] == chrom]
        chrom_se = se_regions_df[se_regions_df['chrom'] == chrom] if len(se_regions_df) > 0 else pd.DataFrame()
        
        for _, enh in chrom_enhancers.iterrows():
            enh_center = (enh['start'] + enh['end']) // 2
            dist = abs(enh_center - tss)
            
            if min_distance <= dist <= max_distance:
                is_se = any((se['start'] <= enh_center <= se['end']) 
                           for _, se in chrom_se.iterrows()) if len(chrom_se) > 0 else False
                
                ep_pairs.append({
                    'gene_name': gene['gene_name'],
                    'ensembl_id': gene['ensembl_id'],
                    'gene_chrom': chrom,
                    'promoter_pos': tss,
                    'enhancer_center': enh_center,
                    'enhancer_signal': enh['signalValue'],
                    'distance': dist,
                    'expression': gene['tpm'],
                    'log2_expression': gene['log2_tpm'],
                    'is_super_enhancer': is_se
                })
    
    ep_df = pd.DataFrame(ep_pairs)
    print(f"Built {len(ep_df)} E-P pairs from {ep_df['gene_name'].nunique()} genes")
    print(f"  Super-enhancer pairs: {ep_df['is_super_enhancer'].sum()}")
    print(f"  Typical enhancer pairs: {(~ep_df['is_super_enhancer']).sum()}")
    return ep_df

ep_df = build_ep_pairs(genes_df, enhancers_df, se_regions_df)
print(f"\nDistance range: {ep_df['distance'].min()/1000:.1f} - {ep_df['distance'].max()/1000:.1f} kb")


---
## Part 4: Extract Contact Frequencies from Micro-C

In [ ]:
def compute_expected_contacts(clr, chromosomes, resolution, max_dist=500000, n_windows=50):
    """Compute distance-dependent expected contacts by sampling random genomic windows."""
    max_bin_dist = max_dist // resolution + 1
    dist_sums   = np.zeros(max_bin_dist + 1)
    dist_counts = np.zeros(max_bin_dist + 1)

    np.random.seed(42)
    chromsizes = clr.chromsizes
    window = 2_000_000

    for chrom in chromosomes:
        chr_len = chromsizes.get(chrom, 0)
        if chr_len < window: continue
        n_win = min(n_windows // max(len(chromosomes), 1) + 1, 10)
        for _ in range(n_win):
            start = np.random.randint(0, chr_len - window)
            try:
                mat = clr.matrix(balance=True).fetch(f'{chrom}:{start}-{start+window}')
                n = mat.shape[0]
                for d in range(1, min(max_bin_dist + 1, n)):
                    diag  = np.diag(mat, d)
                    valid = ~np.isnan(diag) & (diag > 0)
                    dist_sums[d]   += diag[valid].sum()
                    dist_counts[d] += valid.sum()
            except Exception:
                continue

    expected = np.zeros(max_bin_dist + 1)
    for d in range(1, max_bin_dist + 1):
        expected[d] = dist_sums[d] / dist_counts[d] if dist_counts[d] > 10 else np.nan
    return expected


def compute_contacts_for_pairs(ep_df, clr, expected_by_dist, max_dist=500000):
    """Extract balanced Micro-C contacts for all E-P pairs and compute O/E ratios."""
    resolution = clr.binsize
    chromsizes = clr.chromsizes
    obs_dict = {}

    for gene_name, group in ep_df.groupby('gene_name'):
        chrom   = group.iloc[0]['gene_chrom']
        tss     = group.iloc[0]['promoter_pos']
        chr_len = chromsizes.get(chrom, 0)
        if chr_len == 0:
            for idx in group.index:
                obs_dict[idx] = (np.nan, np.nan)
            continue

        region_start = max(0, tss - max_dist - resolution)
        region_end   = min(chr_len, tss + max_dist + resolution)

        try:
            mat = clr.matrix(balance=True).fetch(f'{chrom}:{region_start}-{region_end}')
            offset_bin = region_start // resolution

            for idx, row in group.iterrows():
                bin1 = int(row['promoter_pos']    // resolution - offset_bin)
                bin2 = int(row['enhancer_center'] // resolution - offset_bin)

                if 0 <= bin1 < mat.shape[0] and 0 <= bin2 < mat.shape[1]:
                    obs = mat[bin1, bin2]
                    dist_bins = abs(int(row['enhancer_center'] // resolution) -
                                    int(row['promoter_pos']    // resolution))
                    exp = (expected_by_dist[dist_bins]
                           if dist_bins < len(expected_by_dist) else np.nan)

                    if (not np.isnan(obs) and obs > 0 and
                        not np.isnan(exp) and exp > 0):
                        obs_dict[idx] = (obs, obs / exp)
                    else:
                        obs_dict[idx] = (obs if not np.isnan(obs) else 0, np.nan)
                else:
                    obs_dict[idx] = (np.nan, np.nan)
        except Exception:
            for idx in group.index:
                obs_dict[idx] = (np.nan, np.nan)

    ep_df = ep_df.copy()
    ep_df['observed_contacts'] = ep_df.index.map(
        lambda i: obs_dict.get(i, (np.nan, np.nan))[0])
    ep_df['obs_exp_ratio'] = ep_df.index.map(
        lambda i: obs_dict.get(i, (np.nan, np.nan))[1])
    ep_df['log2_obs_exp'] = np.log2(ep_df['obs_exp_ratio'].replace(0, np.nan))

    valid = ep_df['obs_exp_ratio'].notna() & np.isfinite(ep_df['log2_obs_exp'])
    print(f'Valid pairs: {valid.sum():,} / {len(ep_df):,}')
    return ep_df


if clr is None:
    print('Skipping contact extraction — Micro-C file not loaded.')
else:
    chroms_in_data = ep_df['gene_chrom'].unique().tolist()
    print(f'Computing expected contacts from {len(chroms_in_data)} chromosomes...')
    expected_by_dist = compute_expected_contacts(
        clr, chroms_in_data, clr.binsize, n_windows=100)

    print('Extracting observed contacts...')
    ep_df = compute_contacts_for_pairs(ep_df, clr, expected_by_dist)

    ep_valid = ep_df[
        ep_df['obs_exp_ratio'].notna() & np.isfinite(ep_df['log2_obs_exp'])
    ].copy()
    print(f'\nValid E-P pairs for analysis: {len(ep_valid):,}')


---
## Part 5: Test Goh et al. Predictions

In [ ]:
def test_expression_contact_correlation(ep_df, min_expression=1.0):
    """Test correlation between expression and E-P contacts."""
    df = ep_df[(ep_df['expression'] >= min_expression) & ep_df['obs_exp_ratio'].notna()].copy()
    print(f"Analyzing {len(df)} E-P pairs")
    
    r_pearson, p_pearson = stats.pearsonr(df['log2_expression'], df['log2_obs_exp'])
    r_spearman, p_spearman = stats.spearmanr(df['log2_expression'], df['log2_obs_exp'])
    print(f"Pearson r = {r_pearson:.3f} (p = {p_pearson:.2e})")
    print(f"Spearman rho = {r_spearman:.3f} (p = {p_spearman:.2e})")
    
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(df['log2_expression'], df['log2_obs_exp'], alpha=0.3, s=10, c='steelblue')
    slope, intercept = np.polyfit(df['log2_expression'], df['log2_obs_exp'], 1)
    x_line = np.array([df['log2_expression'].min(), df['log2_expression'].max()])
    ax.plot(x_line, slope * x_line + intercept, 'r-', linewidth=2, label=f'r = {r_pearson:.3f}')
    ax.set_xlabel('log2(TPM + 1)'); ax.set_ylabel('log2(O/E contacts)')
    ax.set_title('Expression vs E-P Contact Frequency'); ax.legend()
    plt.tight_layout(); plt.show()
    
    return {'r_pearson': r_pearson, 'p_pearson': p_pearson, 'r_spearman': r_spearman, 'n_pairs': len(df)}

In [ ]:
def test_distance_dependent_correlation(ep_df, distance_bins=None, min_pairs=50):
    """Test distance-dependence of expression-contact correlation."""
    if distance_bins is None:
        distance_bins = [10000, 25000, 50000, 100000, 200000, 500000]
    
    df = ep_df[(ep_df['expression'] >= 1.0) & ep_df['obs_exp_ratio'].notna()].copy()
    df['distance_bin'] = pd.cut(df['distance'], bins=distance_bins)
    
    results = []
    for bin_label in df['distance_bin'].cat.categories:
        bin_df = df[df['distance_bin'] == bin_label]
        if len(bin_df) >= min_pairs:
            r, p = stats.spearmanr(bin_df['log2_expression'], bin_df['log2_obs_exp'])
            results.append({'distance_center_kb': (bin_label.left + bin_label.right) / 2000,
                           'correlation': r, 'p_value': p, 'n_pairs': len(bin_df)})
    
    results_df = pd.DataFrame(results)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    colors = ['orangered' if p < 0.05 else 'steelblue' for p in results_df['p_value']]
    ax.bar(range(len(results_df)), results_df['correlation'], color=colors, alpha=0.7)
    ax.set_xticks(range(len(results_df)))
    ax.set_xticklabels([f"{r['distance_center_kb']:.0f}kb" for _, r in results_df.iterrows()])
    ax.set_xlabel('Genomic distance'); ax.set_ylabel('Spearman correlation')
    ax.set_title('Distance-Dependent Expression-Contact Correlation (* p < 0.05)')
    ax.axhline(0, color='k', linestyle='--', alpha=0.3)
    plt.tight_layout(); plt.show()
    
    print(results_df.to_string(index=False))
    return results_df

In [ ]:
def compare_super_vs_typical_enhancers(ep_df):
    """Compare expression-contact correlations for super-enhancers vs typical enhancers."""
    df    = ep_df[(ep_df['expression'] >= 1.0) & ep_df['obs_exp_ratio'].notna()].copy()
    se_df = df[df['is_super_enhancer'] == True]
    te_df = df[df['is_super_enhancer'] == False]

    print(f'Super-enhancer pairs: {len(se_df):,},  Typical enhancer pairs: {len(te_df):,}')

    MIN_N = 10
    if len(se_df) >= MIN_N:
        r_se, p_se = stats.spearmanr(se_df['log2_expression'], se_df['log2_obs_exp'])
        if len(se_df) < 30:
            print(f'  NOTE: SE sample small (n={len(se_df)}); interpret correlation cautiously.')
    else:
        r_se, p_se = np.nan, np.nan
        print(f'  SE pairs too few (n={len(se_df)} < {MIN_N}) — correlation not computed.')

    if len(te_df) >= MIN_N:
        r_te, p_te = stats.spearmanr(te_df['log2_expression'], te_df['log2_obs_exp'])
    else:
        r_te, p_te = np.nan, np.nan

    r_se_str = f'{r_se:.3f}' if not np.isnan(r_se) else 'n/a'
    r_te_str = f'{r_te:.3f}' if not np.isnan(r_te) else 'n/a'
    print(f'Super-enhancer Spearman rho = {r_se_str}')
    print(f'Typical enhancer Spearman rho = {r_te_str}')

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].scatter(se_df['log2_expression'], se_df['log2_obs_exp'],
                    alpha=0.5, s=20, c='orangered')
    axes[0].set_title(f'Super-enhancers (\u03c1={r_se_str}, n={len(se_df)})')
    axes[1].scatter(te_df['log2_expression'], te_df['log2_obs_exp'],
                    alpha=0.3, s=10, c='steelblue')
    axes[1].set_title(f'Typical enhancers (\u03c1={r_te_str}, n={len(te_df)})')
    for ax in axes:
        ax.set_xlabel('log$_2$(TPM + 1)')
        ax.set_ylabel('log$_2$(O/E contacts)')
        ax.axhline(0, color='k', linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.show()

    return {'r_se': r_se, 'p_se': p_se, 'r_te': r_te, 'p_te': p_te,
            'n_se': len(se_df), 'n_te': len(te_df)}


---
## Part 6: Run Analysis on Real Data

Run all three tests using the real Micro-C, expression, and H3K27ac data.

**Note:** If the GTF file only covers chr1 (truncated download), run 
`python download_full_gtf.py` from the project root to get genome-wide annotations.

In [ ]:
# Run all tests on REAL data
print("="*60)
print("TEST 1: Overall Expression-Contact Correlation (REAL DATA)")
print("="*60)
results_overall = test_expression_contact_correlation(ep_valid)


In [ ]:
print("="*60)
print("TEST 2: Distance-Dependent Correlation (REAL DATA)")
print("="*60)
results_distance = test_distance_dependent_correlation(ep_valid, min_pairs=15)


In [ ]:
print("="*60)
print("TEST 3: Super-Enhancer vs Typical Enhancer (REAL DATA)")
print("="*60)
results_se = compare_super_vs_typical_enhancers(ep_valid)


In [ ]:
# Generate 4-panel summary figure
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# ---- A: Expression vs Contacts ----
ax = axes[0, 0]
df_plot = ep_valid[ep_valid['expression'] >= 1.0].copy()
ax.scatter(df_plot['log2_expression'], df_plot['log2_obs_exp'],
           alpha=0.05, s=4, c='steelblue', edgecolors='none', rasterized=True)
try:
    bins_p = pd.qcut(df_plot['log2_expression'], q=10, duplicates='drop')
    bp = df_plot.groupby(bins_p, observed=True).agg(
        x=('log2_expression', 'mean'),
        y=('log2_obs_exp',    'mean'),
        ye=('log2_obs_exp',   'sem')).dropna()
    ax.errorbar(bp['x'], bp['y'], yerr=bp['ye'],
                fmt='o-', color='red', markersize=7, linewidth=2,
                capsize=4, zorder=5, label='Binned mean +/- SEM')
    ax.legend(fontsize=9, loc='upper right')
except Exception:
    pass
r_ov, p_ov = stats.spearmanr(df_plot['log2_expression'], df_plot['log2_obs_exp'])
ax.axhline(0, color='k', linestyle='--', alpha=0.4, linewidth=1)
ax.set_xlabel('log$_2$(TPM + 1)', fontsize=12)
ax.set_ylabel('log$_2$(O/E contacts)', fontsize=12)
ax.set_title('A. Expression vs Contacts\n'
             f'rho={r_ov:.3f}, p={p_ov:.2e}, n={len(df_plot):,}', fontsize=11)
ax.set_ylim(-6, 5)

# ---- B: Distance-dependent correlation ----
ax = axes[0, 1]
dist_bins = [5000, 25000, 50000, 100000, 200000, 500000]
df_d = ep_valid[ep_valid['expression'] >= 1.0].copy()
df_d['dbin'] = pd.cut(df_d['distance'], bins=dist_bins)
res = []
for bl in df_d['dbin'].cat.categories:
    bd = df_d[df_d['dbin'] == bl]
    if len(bd) >= 15:
        r, p = stats.spearmanr(bd['log2_expression'], bd['log2_obs_exp'])
        res.append({'bin': f'{bl.left/1000:.0f}-{bl.right/1000:.0f}kb',
                    'rho': r, 'pval': p, 'n': len(bd)})
if res:
    rdf = pd.DataFrame(res)
    bar_colors = ['orangered' if p < 0.05 else 'steelblue' for p in rdf['pval']]
    bars_b = ax.bar(range(len(rdf)), rdf['rho'], color=bar_colors,
                    alpha=0.8, edgecolor='black', linewidth=0.5)
    ax.set_xticks(range(len(rdf)))
    ax.set_xticklabels(rdf['bin'], rotation=25, ha='right', fontsize=10)
    for bar, row in zip(bars_b, rdf.itertuples()):
        if row.pval < 0.001:
            star = '***'
        elif row.pval < 0.01:
            star = '**'
        elif row.pval < 0.05:
            star = '*'
        else:
            star = 'ns'
        ypos = max(row.rho, 0) + 0.001
        ax.text(bar.get_x() + bar.get_width()/2, ypos,
                f'n={row.n:,}\n{star}', ha='center', va='bottom', fontsize=7.5)
    print('Distance-dependent results:')
    print(rdf.to_string(index=False))
ax.axhline(0, color='k', linestyle='--', alpha=0.4, linewidth=1)
ax.set_xlabel('Genomic Distance', fontsize=12)
ax.set_ylabel('Spearman rho', fontsize=12)
ax.set_title('B. Distance-Dependent Correlation\n(orange = p < 0.05)', fontsize=11)

# ---- C: Super vs Typical enhancers ----
ax = axes[1, 0]
se = ep_valid[(ep_valid['expression'] >= 1.0) & (ep_valid['is_super_enhancer'] == True)]
te = ep_valid[(ep_valid['expression'] >= 1.0) & (ep_valid['is_super_enhancer'] == False)]
r_s = stats.spearmanr(se['log2_expression'], se['log2_obs_exp'])[0] if len(se) >= 10 else float('nan')
r_t = stats.spearmanr(te['log2_expression'], te['log2_obs_exp'])[0] if len(te) >= 10 else float('nan')
import math
r_s_val = r_s if not math.isnan(r_s) else 0.0
r_t_val = r_t if not math.isnan(r_t) else 0.0
bars_c = ax.bar(['Super-enhancers', 'Typical'],
                [r_s_val, r_t_val],
                color=['orangered', 'steelblue'],
                alpha=0.8, edgecolor='black', linewidth=0.8, width=0.5)
for bar, rho, n in zip(bars_c, [r_s_val, r_t_val], [len(se), len(te)]):
    bar_mid = bar.get_height() / 2
    ax.text(bar.get_x() + bar.get_width()/2, bar_mid,
            f'rho={rho:.4f}\nn={n:,}',
            ha='center', va='center', fontsize=10,
            color='white', fontweight='bold')
ax.set_ylabel('Spearman rho', fontsize=12)
ax.set_title('C. Super vs Typical Enhancers', fontsize=11)
ax.axhline(0, color='k', linestyle='--', alpha=0.4, linewidth=1)
ax.set_ylim(0, max(r_s_val, r_t_val, 0.005) * 1.8)

# ---- D: Micro-C heatmap (top expressed NUCLEAR gene, in mcool chromosomes) ----
ax = axes[1, 1]
try:
    valid_chroms = set(clr.chromnames)
    nuclear_genes = genes_df[
        (genes_df['tpm'] > 0) &
        (~genes_df['chrom'].isin(['chrM', 'chrMT', 'MT'])) &
        (genes_df['chrom'].isin(valid_chroms))
    ]
    top_gene = nuclear_genes.nlargest(1, 'tpm').iloc[0]
    center   = top_gene['tss']
    win      = 150000
    chr_len  = clr.chromsizes[top_gene['chrom']]
    r_start  = max(0, center - win)
    r_end    = min(chr_len, center + win)
    sub_mat = clr.matrix(balance=True).fetch(
        f"{top_gene['chrom']}:{r_start}-{r_end}")
    im = ax.imshow(
        np.log2(np.nan_to_num(sub_mat, nan=0) + 1),
        cmap='YlOrRd', aspect='auto', origin='upper',
        extent=[r_start/1e6, r_end/1e6, r_end/1e6, r_start/1e6])
    tss_mb = center / 1e6
    ax.axvline(tss_mb, color='blue', linewidth=1.5,
               linestyle='--', alpha=0.9, label=f'TSS ({top_gene["gene_name"]})')
    ax.axhline(tss_mb, color='blue', linewidth=1.5, linestyle='--', alpha=0.9)
    ax.legend(fontsize=9, loc='upper right')
    ax.set_xlabel('Position (Mb)', fontsize=12)
    ax.set_ylabel('Position (Mb)', fontsize=12)
    ax.set_title(f"D. Micro-C: {top_gene['gene_name']} locus\n"
                 f"({top_gene['chrom']}, FPKM={top_gene['tpm']:.0f})", fontsize=11)
    plt.colorbar(im, ax=ax, label='log$_2$(contacts+1)')
except Exception as e:
    ax.text(0.5, 0.5, f'Heatmap error:\n{e}',
            ha='center', va='center', transform=ax.transAxes, fontsize=8)

plt.tight_layout(pad=2.0)
import os
os.makedirs('../images', exist_ok=True)
plt.savefig('../images/analysis_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: analysis_summary.png')


In [ ]:
# Save results
ep_valid.to_csv(DATA_DIR / 'ep_pairs_analyzed.csv', index=False)

# Save individual figures for report
fig1, ax1 = plt.subplots(figsize=(8, 6))
ax1.scatter(df_plot['log2_expression'], df_plot['log2_obs_exp'], alpha=0.15, s=8, c='steelblue', edgecolors='none')
ax1.errorbar(bp['x'], bp['y'], yerr=bp['ye'], fmt='o-', color='red', markersize=8, linewidth=2, capsize=4, label=f'Binned means')
ax1.set_xlabel('log$_2$(TPM + 1)'); ax1.set_ylabel('log$_2$(O/E contacts)')
ax1.set_title('Expression vs E-P Contact Frequency'); ax1.legend()
plt.tight_layout()
fig1.savefig('../images/Expression_vs_EP.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nSaved {len(ep_valid)} E-P pairs to ep_pairs_analyzed.csv")
print(f"Genes: {ep_valid['gene_name'].nunique()}")
print(f"Super-enhancer pairs: {ep_valid['is_super_enhancer'].sum()}")


---
## Summary

### Results from real mESC data:
1. **Expression-contact correlation**: Tests whether higher transcription correlates with more E-P contacts
2. **Distance dependence**: Tests whether the effect is strongest at intermediate distances (50-200kb)
3. **Super-enhancer effect**: Tests whether super-enhancers show stronger expression-contact coupling

### Data sources:
- **Micro-C**: Real mESC chromatin contacts at 5kb resolution
- **Expression**: RSEM-quantified TPM values (ENSMUSG-matched)
- **Enhancers**: H3K27ac ChIP-seq narrowPeaks
- **Super-enhancers**: Top 5% H3K27ac signal, clustered within 12.5kb

### Note on GTF coverage:
If the GENCODE GTF is truncated (chr1 only), results will be limited.
Run `python download_full_gtf.py` from the project root for genome-wide analysis.